In [ ]:
import math

import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)

METHOD = "hmm_pomegranate"


# 1. Read data

In [ ]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/"


In [ ]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = get_available_alleles(DATA_PATH)

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

ALLELES

# 2. Read models

In [ ]:
from hmm_logic_methods import load_model

In [ ]:

RUNS = 15
total_splits = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]])

# 0_run-8-12-st-viterbi-alg-None-batches-1-batch_size-None-init_batch_size-0.0-lr-0.0-edge_inertia-0.0-distribution_inertia-0.0-emission_pseudocount-0.0-transition_pseudocount-30-min_iters-False-use_pseudocount_model-HLA_A_02_01-0
BAUM_WELCH_MODELS_12 = [f"{i}_run-8-12-st-baum-welch-alg-30-min_iters" for i in range(RUNS)]
BAUM_WELCH_MODELS_15 = [f"{i}_run-8-15-st-baum-welch-alg-30-min_iters" for i in range(RUNS)]

BAUM_WELCH_MODELS_20 = [f"{i}_run-8-20-st-baum-welch-alg-30-min_iters" for i in range(RUNS)]

#
#
MODEL_PREFIX_TO_READ =  BAUM_WELCH_MODELS_12 + BAUM_WELCH_MODELS_15  \
                        #BAUM_WELCH_MODELS_20
                       #+ VITERBI_MODELS

per_name_model = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    for model_to_read in MODEL_PREFIX_TO_READ:
            for split_num in range(total_splits):
                model_name = f"{model_to_read}_model-{target_allele_name}-{split_num}"
                model = load_model(TARGET_PATH_TO_RESULTS, model_name)
                per_name_model[model_name] = model
                # print(model_name)



# Populate scores for models

12 states

In [ ]:
models_params_string= "baum-welch-alg-30-min_iters"


def get_models(base_model_prefix, params_string):
    per_allele_per_run_per_split_prepared_models = dict()
    for allele_name in ALLELES:
        per_allele_per_run_per_split_prepared_models[allele_name] = dict()
        for run_num in range(RUNS):
            per_allele_per_run_per_split_prepared_models[allele_name][run_num] = dict()
            for split_num, per_length_data in per_allele_per_kfold_per_length_binders_train[allele_name].items():
                target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
                base_model_name = f"{run_num}_run-{base_model_prefix}-{params_string}_model-{target_allele_name}-{split_num}"

                base_model = per_name_model[base_model_name]
                per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = base_model
    return per_allele_per_run_per_split_prepared_models

base_model_prefix = "8-12-st"

per_allele_per_run_per_split_prepared_models12 = get_models(base_model_prefix, models_params_string)
base_model_prefix = "8-15-st"
per_allele_per_run_per_split_prepared_models15 = get_models(base_model_prefix, models_params_string)

# base_model_prefix = "8-20-st"
# per_allele_per_run_per_split_prepared_models20 = get_models(base_model_prefix, models_params_string)

scores

In [ ]:
import math
target_length = 8
def get_log_scores_df(per_allele_per_run_per_split_models):
    df = pd.DataFrame(columns=[ "allele", "split", "run","train_lp_score","train_vi_score" ,"train_map_score" , "test_lp_score", "test_vi_score", "test_map_score", "model_name","sum_lp_score", "sum_vi_score", "sum_map_score"])

    per_allele_per_split_train_predictions = dict()
    per_allele_per_split_test_predictions = dict()

    for allele_name, per_run_per_split_models in per_allele_per_run_per_split_models.items():
        per_allele_per_split_train_predictions[allele_name] = dict()
        per_allele_per_split_test_predictions[allele_name] = dict()

        for split_num in range(total_splits):
            target_train_data = per_allele_per_kfold_per_length_binders_train[allele_name][split_num][target_length]
            target_test_data = per_allele_per_kfold_per_length_binders_test[allele_name][split_num][target_length]

            per_allele_per_split_train_predictions[allele_name][split_num] = pd.DataFrame({"peptide":target_train_data})
            per_allele_per_split_test_predictions[allele_name][split_num] = pd.DataFrame({"peptide":target_test_data})


        for run_num, per_split_models in per_run_per_split_models.items():
            for split_num, model in per_split_models.items():
                target_train_data = per_allele_per_kfold_per_length_binders_train[allele_name][split_num][target_length]
                target_test_data = per_allele_per_kfold_per_length_binders_test[allele_name][split_num][target_length]
                train_scores_lp =  [model.log_probability(peptide) for peptide in target_train_data]
                train_scores_viterbi =  [model.viterbi(peptide)[0] for peptide in target_train_data]
                train_scores_max_aposteriori = [model.maximum_a_posteriori(list(peptide))[0] for peptide in target_train_data]
                train_sum_lp = sum(train_scores_lp)
                train_sum_vi = sum(train_scores_viterbi)
                train_sum_max_aposteriori = sum(train_scores_max_aposteriori)


                test_scores_lp = [model.log_probability(peptide) for peptide in target_test_data]
                test_scores_viterbi = [model.viterbi(peptide)[0] for peptide in target_test_data]
                test_scores_max_aposteriori = [model.maximum_a_posteriori(list(peptide))[0] for peptide in target_test_data]

                test_sum_lp = sum(test_scores_lp)
                test_sum_vi = sum(test_scores_viterbi)
                test_sum_max_aposteriori = sum(test_scores_max_aposteriori)


                row = {"allele": allele_name,
                       "split": split_num,
                       "run": run_num,
                       "train_lp_score": train_sum_lp,
                       "train_vi_score": train_sum_vi,
                       "train_map_score": train_sum_max_aposteriori,
                       "test_lp_score": test_sum_lp,
                       "test_vi_score": test_sum_vi,
                       "test_map_score": test_sum_max_aposteriori,
                       "sum_lp_score": train_sum_lp + test_sum_lp,
                       "sum_vi_score": train_sum_vi + test_sum_vi,
                       "sum_map_score": train_sum_max_aposteriori + test_sum_max_aposteriori,
                       "model_name": model.name}

                df = df.append(row, ignore_index=True)

                per_allele_per_split_train_predictions[allele_name][split_num][f'score_lp_{run_num}'] = train_scores_lp
                per_allele_per_split_test_predictions[allele_name][split_num][f'score_lp_{run_num}'] = test_scores_lp
                per_allele_per_split_train_predictions[allele_name][split_num][f'score_vi_{run_num}'] = train_scores_viterbi
                per_allele_per_split_test_predictions[allele_name][split_num][f'score_vi_{run_num}'] = test_scores_viterbi
                per_allele_per_split_train_predictions[allele_name][split_num][f'score_map_{run_num}'] = train_scores_max_aposteriori
                per_allele_per_split_test_predictions[allele_name][split_num][f'score_map_{run_num}'] = test_scores_max_aposteriori

        return df, per_allele_per_split_train_predictions, per_allele_per_split_test_predictions

In [ ]:
def get_predictions_and_save_to_files(prefix, per_allele_per_run_per_split_prepared_models):
    sum_predictions,per_allele_per_split_train_predictions, per_allele_per_split_test_predictions  = get_log_scores_df(per_allele_per_run_per_split_prepared_models)
    for allele_name, per_split_predictions in per_allele_per_split_train_predictions.items():
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
        sum_predictions.to_csv(f'{TARGET_PATH_TO_RESULTS}{target_allele_name}_sum_predictions_st{prefix}.csv', index=False)
        for split_num, predictions in per_split_predictions.items():
            predictions.to_csv(f'{TARGET_PATH_TO_RESULTS}{target_allele_name}_st{prefix}_train_split-{split_num}.csv', index=False)

    for allele_name, per_split_predictions in per_allele_per_split_test_predictions.items():
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
        for split_num, predictions in per_split_predictions.items():
            predictions.to_csv(f'{TARGET_PATH_TO_RESULTS}{target_allele_name}_st{prefix}_test_split-{split_num}.csv', index=False)
    return sum_predictions,per_allele_per_split_train_predictions, per_allele_per_split_test_predictions


In [ ]:

# sum_predictions,per_allele_per_split_train_predictions, per_allele_per_split_test_predictions = get_predictions_and_save_to_files("20", per_allele_per_run_per_split_prepared_models20)
sum_predictions,per_allele_per_split_train_predictions, per_allele_per_split_test_predictions = get_predictions_and_save_to_files("15", per_allele_per_run_per_split_prepared_models15)
sum_predictions,per_allele_per_split_train_predictions, per_allele_per_split_test_predictions = get_predictions_and_save_to_files("12", per_allele_per_run_per_split_prepared_models12)